# Task 04 — Item-to-Item Co-Visitation

This notebook is an analytical view of the immutable `task04_item2item_v1` artifact. It imports the reusable implementation and does not refit on raw or validation events.

In [ ]:
import json
from pathlib import Path

import polars as pl

from item2item import Item2ItemConfig, Item2ItemModel, validate_neighbor_table

artifact_dir = Path("artifacts/task04_item2item_v1")
metrics = json.loads((artifact_dir / "metrics.json").read_text(encoding="utf-8"))
resolved_config = json.loads((artifact_dir / "config.json").read_text(encoding="utf-8"))
portable = json.loads((artifact_dir / "model_config.json").read_text(encoding="utf-8"))
neighbors = pl.read_parquet(artifact_dir / "neighbor_table.parquet")
recommendations = pl.read_parquet(artifact_dir / "recommendations.parquet")

In [ ]:
stage_winners = pl.DataFrame(
    [
        {
            "stage": stage["stage"],
            "changed_block": stage["changed_block"],
            "winner_config_id": stage["winner_config_id"],
            "mean_p20_all": stage["summary"][stage["winner_config_id"]]["mean_precision_at_20_all_targets"],
            "mean_p20_labeled": stage["summary"][stage["winner_config_id"]]["mean_precision_at_20_labeled_users"],
        }
        for stage in metrics["selection_stages"]
    ]
)
display(stage_winners)

In [ ]:
canonical_comparison = pl.DataFrame(
    [
        {"run": "task02_global", **{key: metrics["task02_comparison"][key] for key in ("precision_at_20_all_targets", "precision_at_20_labeled_users", "final_hits")}},
        {"run": "task03_recency", **{key: metrics["task03_comparison"][key] for key in ("precision_at_20_all_targets", "precision_at_20_labeled_users", "final_hits")}},
        {"run": "task04_item2item", **{key: metrics["canonical"][key] for key in ("precision_at_20_all_targets", "precision_at_20_labeled_users", "final_hits")}},
    ]
)
display(canonical_comparison)
display(pl.DataFrame([metrics["candidate_hit_overlap"]]))
display(pl.DataFrame([metrics["candidate_union_metrics"]]))

In [ ]:
selected_config = Item2ItemConfig.from_dict(portable["item2item_config"])
validate_neighbor_table(neighbors, neighbor_k=selected_config.neighbor_k)
restored_model = Item2ItemModel.from_fitted_neighbors(selected_config, neighbors)
artifact_checks = {
    "model": restored_model.get_config()["model"],
    "neighbor_rows": neighbors.height,
    "neighbor_items": neighbors.get_column("item_id").n_unique(),
    "recommendation_users": recommendations.height,
    "canonical_evaluated_config_count": metrics["canonical_evaluated_config_count"],
    "deterministic_restore": metrics["deterministic_recommendations_match"],
}
artifact_checks